# Hands-On Case Study | Airline Passengers — RNN vs LSTM vs GRU

**Author:** Dr. Milan Joshi | MathCanvasMJ  
**Sessions:** 4 & 5 — RNN, LSTM, GRU Capstone  
**Framework:** TensorFlow / Keras

> *"The test of all knowledge is experiment."* — Richard Feynman

We've learned three sequence architectures in Sessions 4 and 5. Now it's time for the ultimate showdown: **RNN vs LSTM vs GRU** on a classic time series forecasting problem — the international airline passengers dataset.

### What You'll Learn
1. How to preprocess real-world time series data for deep learning
2. How to build, train, and compare RNN, LSTM, and GRU models
3. Which architecture performs best (and why) on this specific problem
4. How window size affects each architecture differently

## Table of Contents

1. [Quick Recap: RNN vs LSTM vs GRU](#1.-Quick-Recap:-RNN-vs-LSTM-vs-GRU)
2. [The Dataset: International Airline Passengers](#2.-The-Dataset:-International-Airline-Passengers)
3. [Data Preprocessing](#3.-Data-Preprocessing)
4. [Model Definitions — Fair Comparison](#4.-Model-Definitions-—-Fair-Comparison)
5. [Training All Three Models](#5.-Training-All-Three-Models)
6. [Prediction & Evaluation](#6.-Prediction-&-Evaluation)
7. [Window Size Sensitivity Analysis](#7.-Window-Size-Sensitivity-Analysis)
8. [Stacked Models (2-Layer) Comparison](#8.-Stacked-Models-(2-Layer)-Comparison)
9. [Final Analysis & Recommendations](#9.-Final-Analysis-&-Recommendations)

In [ ]:
# ==== Imports and Reproducibility ====
# All three architectures use the same preprocessing, training,
# and evaluation pipeline for a fair comparison.
# ==================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, GRU, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import time
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)
plt.rcParams['figure.dpi'] = 100
print(f"TensorFlow version: {tf.__version__}")

## 1. Quick Recap: RNN vs LSTM vs GRU

### SimpleRNN
$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)$$
- **1 equation**, 1 state vector, params: $n_h(n_x + n_h + 1)$

### LSTM (6 equations, 2 state vectors)
$$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f) \quad \text{(forget gate)}$$
$$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i) \quad \text{(input gate)}$$
$$\tilde{C}_t = \tanh(W_C [h_{t-1}, x_t] + b_C) \quad \text{(candidate)}$$
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t \quad \text{(cell update)}$$
$$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o) \quad \text{(output gate)}$$
$$h_t = o_t \odot \tanh(C_t) \quad \text{(hidden state)}$$
- Params: $4 \times n_h(n_x + n_h + 1)$

### GRU (4 equations, 1 state vector)
$$z_t = \sigma(W_z [h_{t-1}, x_t] + b_z) \quad \text{(update gate)}$$
$$r_t = \sigma(W_r [h_{t-1}, x_t] + b_r) \quad \text{(reset gate)}$$
$$\tilde{h}_t = \tanh(W [r_t \odot h_{t-1}, x_t] + b) \quad \text{(candidate)}$$
$$h_t = (1-z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t \quad \text{(interpolation)}$$
- Params: $3 \times n_h(n_x + n_h + 1)$

In [ ]:
# ==== Triple Architecture Diagram ====
# Side-by-side visual comparison of RNN, LSTM, and GRU cells.
# Each panel shows the internal structure with gate annotations.
# Colors: RNN=#2196F3 (blue), LSTM=#e53935 (red), GRU=#4CAF50 (green)
# ==================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colors = {'RNN': '#2196F3', 'LSTM': '#e53935', 'GRU': '#4CAF50'}

# ---- Panel 1: SimpleRNN Cell ----
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.set_title('SimpleRNN Cell', fontsize=16, fontweight='bold', color=colors['RNN'])
ax.axis('off')

# Main cell box
cell_rnn = plt.Rectangle((2, 3), 6, 4, fill=True, facecolor='#BBDEFB',
                          edgecolor=colors['RNN'], linewidth=2.5, zorder=2)
ax.add_patch(cell_rnn)

# tanh activation block
tanh_box = plt.Rectangle((3.5, 4.2), 3, 1.6, fill=True, facecolor='#E3F2FD',
                          edgecolor=colors['RNN'], linewidth=1.5, zorder=3, linestyle='--')
ax.add_patch(tanh_box)
ax.text(5, 5, 'tanh', ha='center', va='center', fontsize=14, fontweight='bold',
        color=colors['RNN'], zorder=4)

# Input arrow (x_t from bottom)
ax.annotate('', xy=(5, 3), xytext=(5, 1),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.text(5, 0.5, r'$x_t$', ha='center', va='center', fontsize=14, fontweight='bold')

# Output arrow (h_t to top)
ax.annotate('', xy=(5, 9), xytext=(5, 7),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.text(5, 9.5, r'$h_t$', ha='center', va='center', fontsize=14, fontweight='bold')

# Recurrent arrow (h_{t-1} from left)
ax.annotate('', xy=(2, 5), xytext=(0, 5),
            arrowprops=dict(arrowstyle='->', color=colors['RNN'], lw=2))
ax.text(0.5, 5.6, r'$h_{t-1}$', ha='center', va='center', fontsize=12, color=colors['RNN'])

# Recurrent output (h_t to right, loops back)
ax.annotate('', xy=(10, 5), xytext=(8, 5),
            arrowprops=dict(arrowstyle='->', color=colors['RNN'], lw=2))
ax.text(9.5, 5.6, r'$h_t$', ha='center', va='center', fontsize=12, color=colors['RNN'])

# Equation below
ax.text(5, 1.8, r'$h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b)$',
        ha='center', va='center', fontsize=9, style='italic', color='#333')

# ---- Panel 2: LSTM Cell ----
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.set_title('LSTM Cell', fontsize=16, fontweight='bold', color=colors['LSTM'])
ax.axis('off')

# Main cell box
cell_lstm = plt.Rectangle((1, 2), 8, 6, fill=True, facecolor='#FFCDD2',
                           edgecolor=colors['LSTM'], linewidth=2.5, zorder=2)
ax.add_patch(cell_lstm)

# Gate boxes
gate_w, gate_h = 1.4, 0.9
gates_lstm = [
    (1.5, 5.5, r'$f_t$', 'Forget', '#FFEBEE'),
    (3.5, 5.5, r'$i_t$', 'Input', '#FFEBEE'),
    (5.5, 5.5, r'$\tilde{C}_t$', 'Cand.', '#FFEBEE'),
    (7.5, 5.5, r'$o_t$', 'Output', '#FFEBEE'),
]
for gx, gy, label, name, gc in gates_lstm:
    box = plt.Rectangle((gx, gy), gate_w, gate_h, fill=True, facecolor=gc,
                         edgecolor=colors['LSTM'], linewidth=1.5, zorder=3)
    ax.add_patch(box)
    ax.text(gx + gate_w/2, gy + gate_h/2, label, ha='center', va='center',
            fontsize=11, fontweight='bold', color=colors['LSTM'], zorder=4)
    ax.text(gx + gate_w/2, gy - 0.3, name, ha='center', va='center',
            fontsize=7, color='#666', zorder=4)

# Cell state highway (top arrow)
ax.annotate('', xy=(9, 8.5), xytext=(1, 8.5),
            arrowprops=dict(arrowstyle='->', color=colors['LSTM'], lw=2.5, linestyle='--'))
ax.text(5, 9, r'Cell State Highway: $C_{t-1} \rightarrow C_t$',
        ha='center', va='center', fontsize=10, fontweight='bold', color=colors['LSTM'])

# Hidden state arrows
ax.text(5, 3, r'$h_t = o_t \odot \tanh(C_t)$', ha='center', va='center',
        fontsize=9, style='italic', color='#333', zorder=4)

# Input / output
ax.annotate('', xy=(5, 2), xytext=(5, 0.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.text(5, 0.2, r'$x_t$', ha='center', va='center', fontsize=14, fontweight='bold')

# h_{t-1}
ax.annotate('', xy=(1, 4), xytext=(0, 4),
            arrowprops=dict(arrowstyle='->', color=colors['LSTM'], lw=2))
ax.text(0, 4.5, r'$h_{t-1}$', ha='center', fontsize=10, color=colors['LSTM'])

# State label
ax.text(5, 7.5, '2 States: $C_t$ (cell) + $h_t$ (hidden)',
        ha='center', va='center', fontsize=9, fontweight='bold',
        color=colors['LSTM'], zorder=4,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

# ---- Panel 3: GRU Cell ----
ax = axes[2]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.set_title('GRU Cell', fontsize=16, fontweight='bold', color=colors['GRU'])
ax.axis('off')

# Main cell box
cell_gru = plt.Rectangle((1.5, 2.5), 7, 5, fill=True, facecolor='#C8E6C9',
                          edgecolor=colors['GRU'], linewidth=2.5, zorder=2)
ax.add_patch(cell_gru)

# Gate boxes
gates_gru = [
    (2, 5.5, r'$z_t$', 'Update', '#E8F5E9'),
    (4.5, 5.5, r'$r_t$', 'Reset', '#E8F5E9'),
    (7, 5.5, r'$\tilde{h}_t$', 'Cand.', '#E8F5E9'),
]
for gx, gy, label, name, gc in gates_gru:
    box = plt.Rectangle((gx, gy), gate_w, gate_h, fill=True, facecolor=gc,
                         edgecolor=colors['GRU'], linewidth=1.5, zorder=3)
    ax.add_patch(box)
    ax.text(gx + gate_w/2, gy + gate_h/2, label, ha='center', va='center',
            fontsize=11, fontweight='bold', color=colors['GRU'], zorder=4)
    ax.text(gx + gate_w/2, gy - 0.3, name, ha='center', va='center',
            fontsize=7, color='#666', zorder=4)

# Interpolation equation
ax.text(5, 3.5, r'$h_t = (1-z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$',
        ha='center', va='center', fontsize=9, style='italic', color='#333', zorder=4)

# Input / output
ax.annotate('', xy=(5, 2.5), xytext=(5, 1),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.text(5, 0.5, r'$x_t$', ha='center', va='center', fontsize=14, fontweight='bold')

ax.annotate('', xy=(5, 9.5), xytext=(5, 7.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.text(5, 9.8, r'$h_t$', ha='center', va='center', fontsize=14, fontweight='bold')

# h_{t-1}
ax.annotate('', xy=(1.5, 5), xytext=(0, 5),
            arrowprops=dict(arrowstyle='->', color=colors['GRU'], lw=2))
ax.text(0, 5.5, r'$h_{t-1}$', ha='center', fontsize=10, color=colors['GRU'])

# State label
ax.text(5, 8.2, '1 State: $h_t$ only (no cell state)',
        ha='center', va='center', fontsize=9, fontweight='bold',
        color=colors['GRU'], zorder=4,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

plt.suptitle('Architecture Comparison: RNN vs LSTM vs GRU', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 2. The Dataset: International Airline Passengers

The **Box-Jenkins airline passenger dataset** is one of the most famous time series in statistics. It records the total number of international airline passengers (in thousands) per month from January 1949 to December 1960.

**Key characteristics:**
- **Trend**: Clear upward trend (air travel was booming post-WW2)
- **Seasonality**: Strong 12-month seasonal pattern (summer peaks)
- **Increasing variance**: The seasonal fluctuations grow with the level (multiplicative seasonality)
- **144 data points**: Small dataset — perfect for testing model efficiency

In [ ]:
# ==== Load and Inspect the Dataset ====
# The airline-passengers.csv sits in the same directory as this notebook.
# 144 rows: monthly passenger counts from Jan 1949 to Dec 1960.
# ==================================================
df = pd.read_csv('airline-passengers.csv')

# Parse dates and set as index for proper time series handling
df['Month'] = pd.to_datetime(df['Month'])
df.set_index('Month', inplace=True)

print("=" * 50)
print("DATASET INFO")
print("=" * 50)
print(f"Shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"Frequency: Monthly")
print()

print("--- First 5 Rows ---")
print(df.head())
print()

print("--- Last 5 Rows ---")
print(df.tail())
print()

print("--- Statistical Summary ---")
print(df.describe())
print()

print("--- Data Types ---")
df.info()

In [ ]:
# ==== EDA Plot 1: Raw Time Series with Annotations ====
# Visualize the full time series with trend line and seasonal peak annotations.
# This reveals the key characteristics: upward trend + multiplicative seasonality.
# ==================================================
fig, ax = plt.subplots(figsize=(14, 5))

# Plot raw data
ax.plot(df.index, df['Passengers'], color='#1976D2', linewidth=1.5, label='Monthly Passengers')

# Trend line (12-month rolling average)
trend = df['Passengers'].rolling(window=12, center=True).mean()
ax.plot(df.index, trend, color='#e53935', linewidth=2.5, linestyle='--', label='12-Month Trend')

# Annotate summer peaks (July of each year)
for year in range(1949, 1961):
    july_idx = df.index[(df.index.month == 7) & (df.index.year == year)]
    if len(july_idx) > 0:
        val = df.loc[july_idx, 'Passengers'].values[0]
        if year in [1949, 1954, 1960]:  # Annotate select years to avoid clutter
            ax.annotate(f'Jul {year}\n{val}k',
                       xy=(july_idx[0], val),
                       xytext=(0, 20), textcoords='offset points',
                       fontsize=8, ha='center', color='#e53935',
                       arrowprops=dict(arrowstyle='->', color='#e53935', lw=1))
        ax.plot(july_idx[0], val, 'v', color='#e53935', markersize=5, alpha=0.6)

ax.set_title('International Airline Passengers (1949-1960)', fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Passengers (thousands)', fontsize=12)
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_facecolor('#FAFAFA')
plt.tight_layout()
plt.show()

In [ ]:
# ==== EDA Plot 2: Seasonal Decomposition (Manual with NumPy) ====
# We decompose the series into Trend, Seasonal, and Residual components
# using a 12-month centered rolling average approach.
# This is the multiplicative decomposition: Y = T * S * R
# ==================================================

passengers = df['Passengers'].values.astype(float)
dates = df.index

# Step 1: Trend — 12-month centered moving average
trend_vals = df['Passengers'].rolling(window=12, center=True).mean().values

# Step 2: Detrended = original / trend (multiplicative)
detrended = passengers / trend_vals

# Step 3: Seasonal — average detrended value for each month (1-12)
months = df.index.month
seasonal_indices = np.zeros(12)
for m in range(1, 13):
    mask = (months == m) & (~np.isnan(detrended))
    seasonal_indices[m-1] = np.nanmean(detrended[mask])

# Normalize seasonal indices to average 1.0
seasonal_indices = seasonal_indices / seasonal_indices.mean()
seasonal_component = np.array([seasonal_indices[m-1] for m in months])

# Step 4: Residual = original / (trend * seasonal)
residual = passengers / (trend_vals * seasonal_component)

# Plot 3-panel decomposition
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Panel 1: Trend
axes[0].plot(dates, passengers, color='#90CAF9', alpha=0.5, linewidth=1, label='Original')
axes[0].plot(dates, trend_vals, color='#1976D2', linewidth=2.5, label='Trend (12-mo avg)')
axes[0].set_ylabel('Passengers', fontsize=12)
axes[0].set_title('Trend Component', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Panel 2: Seasonal
axes[1].plot(dates, seasonal_component, color='#4CAF50', linewidth=1.5)
axes[1].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Seasonal Index', fontsize=12)
axes[1].set_title('Seasonal Component (Multiplicative)', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].text(dates[3], max(seasonal_component)*0.98,
             'Above 1.0 = above-average months (summer)',
             fontsize=9, color='#4CAF50', style='italic')

# Panel 3: Residual
axes[2].plot(dates, residual, color='#FF9800', linewidth=1, alpha=0.8)
axes[2].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
axes[2].set_ylabel('Residual', fontsize=12)
axes[2].set_xlabel('Date', fontsize=12)
axes[2].set_title('Residual Component', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Seasonal Decomposition of Airline Passengers', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ==== EDA Plot 3: Year-over-Year Overlay ====
# Plot each year's monthly data as a separate line on the same axes.
# This clearly reveals the seasonal pattern and how it grows over time.
# ==================================================

fig, ax = plt.subplots(figsize=(12, 6))

month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# Color gradient from light to dark blue across years
cmap = plt.cm.Blues
years = range(1949, 1961)
n_years = len(list(years))

for i, year in enumerate(years):
    yearly_data = df[df.index.year == year]['Passengers'].values
    color = cmap(0.3 + 0.7 * i / (n_years - 1))
    alpha = 0.5 + 0.5 * i / (n_years - 1)
    lw = 1.0 + 1.5 * i / (n_years - 1)
    ax.plot(range(1, len(yearly_data) + 1), yearly_data,
            color=color, linewidth=lw, alpha=alpha,
            label=str(year), marker='o', markersize=3)

ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels, fontsize=11)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Passengers (thousands)', fontsize=12)
ax.set_title('Year-over-Year Seasonal Pattern (1949-1960)', fontsize=16, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9, title='Year', title_fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_facecolor('#FAFAFA')

# Annotate the growing amplitude
ax.annotate('Seasonal amplitude\ngrows each year',
            xy=(7, 600), fontsize=10, color='#e53935', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFEBEE', alpha=0.8))

plt.tight_layout()
plt.show()

## 3. Data Preprocessing

Time series preprocessing for deep learning has specific requirements:

| Step | Why |
|------|-----|
| **MinMaxScaler [0,1]** | Neural networks train faster with normalized inputs |
| **Sliding window** | Convert a single sequence into supervised learning pairs (X, y) |
| **Time-based split** | NEVER shuffle time series — future data must not leak into training |
| **3D reshape** | Keras RNN layers expect `[samples, timesteps, features]` |

### The Sliding Window Approach
Given a window of size $w$, we create:
$$X_i = [s_{i}, s_{i+1}, \ldots, s_{i+w-1}], \quad y_i = s_{i+w}$$

where $s_t$ is the scaled value at time $t$.

In [ ]:
# ==== Data Preprocessing Pipeline ====
# 1. MinMaxScaler to [0, 1] — essential for neural networks
# 2. Create sliding window sequences (window=12 for 1 year lookback)
# 3. 70/30 time-based split (NEVER random for time series!)
# 4. Reshape to 3D: [samples, timesteps, features]
# ==================================================
values = df['Passengers'].values.astype('float32').reshape(-1, 1)
scaler = MinMaxScaler()
scaled = scaler.fit_transform(values).flatten()

def create_sequences(data, window_size):
    """Convert a flat time series into (X, y) pairs using a sliding window.
    
    Parameters:
        data: 1D array of scaled values
        window_size: number of past time steps to use as input
    
    Returns:
        X: array of shape (n_samples, window_size)
        y: array of shape (n_samples,)
    """
    X, y = [], []
    for i in range(window_size, len(data)):
        X.append(data[i-window_size:i])
        y.append(data[i])
    return np.array(X), np.array(y)

WINDOW = 12  # 12 months lookback (1 full seasonal cycle)
X, y = create_sequences(scaled, WINDOW)

# Time-based split: 70% train, 30% test
split = int(0.7 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Reshape to 3D: [samples, timesteps, features] — required by Keras RNN layers
X_train = X_train.reshape(-1, WINDOW, 1)
X_test = X_test.reshape(-1, WINDOW, 1)

print(f"Total samples:  {len(X)}")
print(f"Training set:   X={X_train.shape}, y={y_train.shape}")
print(f"Test set:       X={X_test.shape}, y={y_test.shape}")
print(f"Window size:    {WINDOW} months (1 year lookback)")
print(f"Split point:    Index {split} (70/30 split)")
print(f"Scaler range:   [{scaler.data_min_[0]:.0f}, {scaler.data_max_[0]:.0f}] -> [0, 1]")

In [ ]:
# ==== Train/Test Split Visualization ====
# Time series must be split chronologically, never randomly.
# This plot shows exactly where the split occurs.
# ==================================================

fig, ax = plt.subplots(figsize=(14, 5))

# We need the dates corresponding to our X/y arrays
# X[i] uses data[i:i+WINDOW], y[i] = data[i+WINDOW]
# So the date for y[i] is df.index[i + WINDOW]
all_dates = df.index[WINDOW:]
train_dates = all_dates[:split]
test_dates = all_dates[split:]

# Inverse-transform y values for plotting in original scale
y_train_orig = scaler.inverse_transform(y_train.reshape(-1, 1)).flatten()
y_test_orig = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

ax.plot(train_dates, y_train_orig, color='#1976D2', linewidth=1.8, label=f'Training ({len(y_train)} samples)')
ax.plot(test_dates, y_test_orig, color='#e53935', linewidth=1.8, label=f'Test ({len(y_test)} samples)')

# Vertical split line
split_date = all_dates[split]
ax.axvline(x=split_date, color='black', linestyle='--', linewidth=2, alpha=0.7)
ax.text(split_date, max(y_test_orig) * 1.02, f'Split: {split_date.strftime("%b %Y")}',
        ha='center', va='bottom', fontsize=11, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.8))

# Shade regions
ax.axvspan(train_dates[0], split_date, alpha=0.05, color='#1976D2')
ax.axvspan(split_date, test_dates[-1], alpha=0.05, color='#e53935')

ax.set_title('Train/Test Split (70/30 — Time-Based)', fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Passengers (thousands)', fontsize=12)
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Model Definitions — Fair Comparison

For a fair comparison, all three models use **identical hyperparameters**:
- **Units:** 64
- **Optimizer:** Adam (lr=0.001)
- **Loss:** Mean Squared Error
- **Epochs:** 200 (with EarlyStopping patience=20)
- **Batch size:** 16
- **Architecture:** Single recurrent layer $\rightarrow$ Dense(1)

The **only** variable across models is the recurrent layer type: `SimpleRNN`, `LSTM`, or `GRU`.

### Parameter Count Comparison

For $n_h = 64$ hidden units and $n_x = 1$ input feature:

| Model | Formula | Recurrent Params | Dense Params | Total |
|-------|---------|-----------------|-------------|-------|
| SimpleRNN | $n_h(n_x + n_h + 1)$ | $64(1+64+1) = 4{,}224$ | $64+1 = 65$ | **4,289** |
| LSTM | $4 \times n_h(n_x + n_h + 1)$ | $4 \times 4{,}224 = 16{,}896$ | 65 | **16,961** |
| GRU | $3 \times n_h(n_x + n_h + 1)$ | $3 \times 4{,}224 = 12{,}672$ | 65 | **12,737** |

In [ ]:
# ==== Model Builder Functions ====
# Each function creates an identical architecture except for the recurrent layer.
# Single recurrent layer (units) -> Dense(1) for regression output.
# ==================================================

def build_rnn(units=64, window=12):
    """Build a SimpleRNN model for time series forecasting."""
    model = Sequential([
        SimpleRNN(units, input_shape=(window, 1)),
        Dense(1)
    ])
    model.compile(optimizer=Adam(0.001), loss='mse')
    return model

def build_lstm(units=64, window=12):
    """Build an LSTM model for time series forecasting."""
    model = Sequential([
        LSTM(units, input_shape=(window, 1)),
        Dense(1)
    ])
    model.compile(optimizer=Adam(0.001), loss='mse')
    return model

def build_gru(units=64, window=12):
    """Build a GRU model for time series forecasting."""
    model = Sequential([
        GRU(units, input_shape=(window, 1)),
        Dense(1)
    ])
    model.compile(optimizer=Adam(0.001), loss='mse')
    return model

print("Model builder functions defined: build_rnn(), build_lstm(), build_gru()")

In [ ]:
# ==== Model Summaries & Parameter Count Comparison ====
# Print the summary for each model and create a visual comparison
# of parameter counts across the three architectures.
# ==================================================

model_names = ['SimpleRNN', 'LSTM', 'GRU']
builders = [build_rnn, build_lstm, build_gru]
param_counts = []

for name, builder in zip(model_names, builders):
    print(f"\n{'='*60}")
    print(f"  {name} Architecture")
    print(f"{'='*60}")
    model = builder(units=64, window=WINDOW)
    model.summary()
    param_counts.append(model.count_params())
    del model

# Parameter count bar chart
fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = ['#2196F3', '#e53935', '#4CAF50']
bars = ax.bar(model_names, param_counts, color=bar_colors, edgecolor='white', linewidth=2, width=0.5)

# Add value labels on bars
for bar, count in zip(bars, param_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{count:,}', ha='center', va='bottom', fontsize=13, fontweight='bold')

# Add multiplier annotations
ax.text(0, param_counts[0]/2, '1x', ha='center', va='center',
        fontsize=16, fontweight='bold', color='white')
ax.text(1, param_counts[1]/2, '~4x', ha='center', va='center',
        fontsize=16, fontweight='bold', color='white')
ax.text(2, param_counts[2]/2, '~3x', ha='center', va='center',
        fontsize=16, fontweight='bold', color='white')

ax.set_ylabel('Total Parameters', fontsize=13)
ax.set_title('Parameter Count Comparison: RNN vs LSTM vs GRU', fontsize=15, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_facecolor('#FAFAFA')
plt.tight_layout()
plt.show()

print(f"\nLSTM has {param_counts[1]/param_counts[0]:.1f}x more params than SimpleRNN")
print(f"GRU has {param_counts[2]/param_counts[0]:.1f}x more params than SimpleRNN")
print(f"GRU has {param_counts[2]/param_counts[1]*100:.0f}% of LSTM's params")

## 5. Training All Three Models

We train all three models under identical conditions:
- Same data, same split, same hyperparameters
- **EarlyStopping** with `patience=20` and `restore_best_weights=True`
- 15% of training data used for validation

Each model is evaluated on the **held-out test set** (the last 30% of the time series).

### Metrics We Track
| Metric | Formula | Interpretation |
|--------|---------|---------------|
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ | Average error magnitude (same units as data) |
| **MAE** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Average absolute error (robust to outliers) |
| **R$^2$** | $1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$ | Proportion of variance explained (1.0 = perfect) |
| **MAPE** | $\frac{100}{n}\sum\left|\frac{y_i - \hat{y}_i}{y_i}\right|$ | Percentage error (scale-independent) |

In [ ]:
# ==== Train All Three Models ====
# Each model trains under identical conditions for fair comparison.
# We track: training history, training time, epochs completed.
# Predictions are inverse-transformed back to original passenger scale.
# ==================================================
results = {}

for name, builder in [('RNN', build_rnn), ('LSTM', build_lstm), ('GRU', build_gru)]:
    print(f"\n{'='*50}")
    print(f"Training {name}...")
    print(f"{'='*50}")
    
    # Reset seeds for each model for reproducibility
    np.random.seed(42)
    tf.random.set_seed(42)
    
    model = builder(units=64, window=WINDOW)
    
    start_time = time.time()
    history = model.fit(
        X_train, y_train,
        epochs=200,
        batch_size=16,
        validation_split=0.15,
        callbacks=[EarlyStopping(patience=20, restore_best_weights=True)],
        verbose=0
    )
    train_time = time.time() - start_time
    
    # Generate predictions on both train and test sets
    pred_train = model.predict(X_train, verbose=0).flatten()
    pred_test = model.predict(X_test, verbose=0).flatten()
    
    # Inverse transform predictions and actuals back to original scale
    pred_train_inv = scaler.inverse_transform(pred_train.reshape(-1, 1)).flatten()
    pred_test_inv = scaler.inverse_transform(pred_test.reshape(-1, 1)).flatten()
    y_train_inv = scaler.inverse_transform(y_train.reshape(-1, 1)).flatten()
    y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
    
    # Compute evaluation metrics on test set (original scale)
    rmse = np.sqrt(mean_squared_error(y_test_inv, pred_test_inv))
    mae = mean_absolute_error(y_test_inv, pred_test_inv)
    r2 = r2_score(y_test_inv, pred_test_inv)
    mape = np.mean(np.abs((y_test_inv - pred_test_inv) / y_test_inv)) * 100
    
    # Store everything
    results[name] = {
        'history': history,
        'model': model,
        'pred_train': pred_train_inv,
        'pred_test': pred_test_inv,
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'mape': mape,
        'time': train_time,
        'epochs': len(history.history['loss']),
        'params': model.count_params()
    }
    
    print(f"  Epochs completed: {results[name]['epochs']}")
    print(f"  Training time:    {train_time:.1f}s")
    print(f"  Test RMSE:        {rmse:.2f}")
    print(f"  Test MAE:         {mae:.2f}")
    print(f"  Test R²:          {r2:.4f}")
    print(f"  Test MAPE:        {mape:.1f}%")

print(f"\n{'='*50}")
print("All three models trained successfully!")
print(f"{'='*50}")

In [ ]:
# ==== Training Curves: Loss vs Epochs ====
# Overlaid training and validation loss curves for all 3 models.
# Solid lines = training loss, dashed lines = validation loss.
# Helps diagnose convergence speed and overfitting.
# ==================================================

fig, ax = plt.subplots(figsize=(14, 6))

model_colors = {'RNN': '#2196F3', 'LSTM': '#e53935', 'GRU': '#4CAF50'}

for name in ['RNN', 'LSTM', 'GRU']:
    hist = results[name]['history'].history
    epochs_range = range(1, len(hist['loss']) + 1)
    color = model_colors[name]
    
    # Training loss (solid)
    ax.plot(epochs_range, hist['loss'], color=color, linewidth=2,
            label=f'{name} Train', alpha=0.9)
    
    # Validation loss (dashed)
    ax.plot(epochs_range, hist['val_loss'], color=color, linewidth=2,
            linestyle='--', label=f'{name} Val', alpha=0.7)

ax.set_xlabel('Epoch', fontsize=13)
ax.set_ylabel('Loss (MSE)', fontsize=13)
ax.set_title('Training & Validation Loss — RNN vs LSTM vs GRU', fontsize=16, fontweight='bold')
ax.legend(fontsize=10, ncol=3, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_facecolor('#FAFAFA')
ax.set_yscale('log')  # Log scale to see differences at low loss values
ax.set_ylabel('Loss (MSE, log scale)', fontsize=13)

# Annotate early stopping
for name in ['RNN', 'LSTM', 'GRU']:
    ep = results[name]['epochs']
    final_val = results[name]['history'].history['val_loss'][-1]
    ax.annotate(f'{name}: {ep} ep',
               xy=(ep, final_val), fontsize=8, fontweight='bold',
               color=model_colors[name],
               xytext=(5, 10), textcoords='offset points')

plt.tight_layout()
plt.show()

## 6. Prediction & Evaluation

Now we visually compare the predictions from all three models against the actual passenger counts. The key question:

> **Which architecture best captures the trend AND seasonal patterns?**

We'll look at:
1. Full timeline predictions (train + test)
2. Zoomed-in test set predictions
3. Quantitative metrics comparison

In [ ]:
# ==== Main Prediction Plot: Full Timeline ====
# Actual data in black, with each model's predictions overlaid.
# Vertical dashed line marks the train/test boundary.
# ==================================================

fig, ax = plt.subplots(figsize=(16, 8))

# Actual values
ax.plot(train_dates, y_train_inv, color='black', linewidth=2, label='Actual (Train)', alpha=0.8)
ax.plot(test_dates, y_test_inv, color='black', linewidth=2.5, label='Actual (Test)',
        linestyle='-', marker='o', markersize=3)

# Model predictions
for name, color, ls in [('RNN', '#2196F3', '-'), ('LSTM', '#e53935', '-'), ('GRU', '#4CAF50', '-')]:
    ax.plot(train_dates, results[name]['pred_train'], color=color,
            linewidth=1.2, alpha=0.5)  # Train predictions (faded)
    ax.plot(test_dates, results[name]['pred_test'], color=color,
            linewidth=2, label=f'{name} (RMSE={results[name]["rmse"]:.1f})',
            linestyle=ls, marker='s', markersize=3)

# Train/test split line
ax.axvline(x=split_date, color='black', linestyle='--', linewidth=2, alpha=0.5)
ax.text(split_date, ax.get_ylim()[1] * 0.95, ' Test Region \u2192',
        fontsize=11, fontweight='bold', color='#e53935')
ax.text(split_date, ax.get_ylim()[1] * 0.95, '\u2190 Train Region ',
        fontsize=11, fontweight='bold', color='#1976D2', ha='right')

ax.set_title('Airline Passenger Predictions — RNN vs LSTM vs GRU', fontsize=18, fontweight='bold')
ax.set_xlabel('Date', fontsize=13)
ax.set_ylabel('Passengers (thousands)', fontsize=13)
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_facecolor('#FAFAFA')
plt.tight_layout()
plt.show()

In [ ]:
# ==== Zoomed Test Set Predictions ====
# Enlarged view of just the test portion to see differences clearly.
# Error bands highlight where each model deviates from actual.
# ==================================================

fig, ax = plt.subplots(figsize=(14, 7))

# Actual test values
ax.plot(test_dates, y_test_inv, color='black', linewidth=2.5,
        label='Actual', marker='o', markersize=5, zorder=5)

# Each model's test predictions
for name, color, marker in [('RNN', '#2196F3', 's'), ('LSTM', '#e53935', '^'), ('GRU', '#4CAF50', 'D')]:
    pred = results[name]['pred_test']
    ax.plot(test_dates, pred, color=color, linewidth=2,
            label=f'{name} (RMSE={results[name]["rmse"]:.1f}, R²={results[name]["r2"]:.3f})',
            marker=marker, markersize=5, alpha=0.85)
    
    # Fill between actual and predicted to visualize errors
    ax.fill_between(test_dates, y_test_inv, pred, alpha=0.1, color=color)

ax.set_title('Test Set Predictions (Zoomed) — RNN vs LSTM vs GRU', fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=13)
ax.set_ylabel('Passengers (thousands)', fontsize=13)
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_facecolor('#FAFAFA')
plt.tight_layout()
plt.show()

In [ ]:
# ==== Metrics Comparison: Grouped Bar Chart ====
# 4 metric groups (RMSE, MAE, R2, MAPE) with 3 bars each (RNN, LSTM, GRU).
# Also print a clean metrics DataFrame.
# ==================================================

metrics_data = {
    'RMSE': [results[m]['rmse'] for m in ['RNN', 'LSTM', 'GRU']],
    'MAE': [results[m]['mae'] for m in ['RNN', 'LSTM', 'GRU']],
    'R²': [results[m]['r2'] for m in ['RNN', 'LSTM', 'GRU']],
    'MAPE (%)': [results[m]['mape'] for m in ['RNN', 'LSTM', 'GRU']],
}

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
bar_colors = ['#2196F3', '#e53935', '#4CAF50']
model_labels = ['RNN', 'LSTM', 'GRU']

for idx, (metric_name, values_list) in enumerate(metrics_data.items()):
    ax = axes[idx]
    bars = ax.bar(model_labels, values_list, color=bar_colors, edgecolor='white', linewidth=2, width=0.5)
    
    # Value labels
    for bar, val in zip(bars, values_list):
        fmt = f'{val:.4f}' if metric_name == 'R²' else f'{val:.1f}'
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                fmt, ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    ax.set_title(metric_name, fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.set_facecolor('#FAFAFA')
    
    # Highlight best
    if metric_name == 'R²':
        best_idx = np.argmax(values_list)
    else:
        best_idx = np.argmin(values_list)
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(3)

plt.suptitle('Test Set Metrics Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Print metrics as a clean DataFrame
print("\n" + "="*60)
print("METRICS COMPARISON TABLE")
print("="*60)
metrics_df = pd.DataFrame({
    'Model': ['RNN', 'LSTM', 'GRU'],
    'RMSE': [f"{results[m]['rmse']:.2f}" for m in ['RNN', 'LSTM', 'GRU']],
    'MAE': [f"{results[m]['mae']:.2f}" for m in ['RNN', 'LSTM', 'GRU']],
    'R²': [f"{results[m]['r2']:.4f}" for m in ['RNN', 'LSTM', 'GRU']],
    'MAPE (%)': [f"{results[m]['mape']:.1f}" for m in ['RNN', 'LSTM', 'GRU']],
})
print(metrics_df.to_string(index=False))

In [ ]:
# ==== Training Time & Parameter Count Comparison ====
# 2-panel bar chart: (1) training time in seconds, (2) total parameters.
# Shows the computational cost of each architecture.
# ==================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
bar_colors = ['#2196F3', '#e53935', '#4CAF50']
model_labels = ['RNN', 'LSTM', 'GRU']

# Panel 1: Training Time
times = [results[m]['time'] for m in model_labels]
bars1 = ax1.bar(model_labels, times, color=bar_colors, edgecolor='white', linewidth=2, width=0.5)
for bar, t in zip(bars1, times):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{t:.1f}s', ha='center', va='bottom', fontsize=13, fontweight='bold')
ax1.set_ylabel('Training Time (seconds)', fontsize=12)
ax1.set_title('Training Time', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
ax1.set_facecolor('#FAFAFA')

# Add epochs annotation
for i, name in enumerate(model_labels):
    ax1.text(i, times[i] * 0.5, f"{results[name]['epochs']} ep",
             ha='center', va='center', fontsize=11, color='white', fontweight='bold')

# Panel 2: Parameter Count
params = [results[m]['params'] for m in model_labels]
bars2 = ax2.bar(model_labels, params, color=bar_colors, edgecolor='white', linewidth=2, width=0.5)
for bar, p in zip(bars2, params):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
             f'{p:,}', ha='center', va='bottom', fontsize=13, fontweight='bold')
ax2.set_ylabel('Total Parameters', fontsize=12)
ax2.set_title('Model Size (Parameters)', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
ax2.set_facecolor('#FAFAFA')

plt.suptitle('Computational Cost: RNN vs LSTM vs GRU', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Window Size Sensitivity Analysis

The lookback window determines how far into the past each model can see. Different architectures may prefer different window sizes:

- **RNN**: May struggle with large windows (vanishing gradients)
- **LSTM**: Should handle large windows well (cell state highway)
- **GRU**: Moderate — between RNN and LSTM

### Hypothesis
- For windows $< 12$ (less than one seasonal cycle), all models should struggle
- For windows $\approx 12$, performance should be good (captures one full seasonal cycle)
- For windows $> 24$, RNN performance may degrade while LSTM remains stable

In [ ]:
# ==== Window Size Sensitivity Experiment ====
# Test all 3 architectures across multiple lookback windows.
# This reveals how well each handles short vs long memory requirements.
# ==================================================

window_sizes = [3, 6, 12, 24, 36]
window_results = {name: [] for name in ['RNN', 'LSTM', 'GRU']}

print("Window Size Sensitivity Analysis")
print("=" * 60)

for ws in window_sizes:
    print(f"\nWindow = {ws} months:")
    
    # Rebuild sequences with new window size
    X_ws, y_ws = create_sequences(scaled, ws)
    split_ws = int(0.7 * len(X_ws))
    X_tr_ws = X_ws[:split_ws].reshape(-1, ws, 1)
    X_te_ws = X_ws[split_ws:].reshape(-1, ws, 1)
    y_tr_ws = y_ws[:split_ws]
    y_te_ws = y_ws[split_ws:]
    
    for name, builder in [('RNN', build_rnn), ('LSTM', build_lstm), ('GRU', build_gru)]:
        np.random.seed(42)
        tf.random.set_seed(42)
        
        model = builder(units=64, window=ws)
        model.fit(X_tr_ws, y_tr_ws, epochs=200, batch_size=16,
                  validation_split=0.15,
                  callbacks=[EarlyStopping(patience=20, restore_best_weights=True)],
                  verbose=0)
        
        pred = model.predict(X_te_ws, verbose=0).flatten()
        pred_inv = scaler.inverse_transform(pred.reshape(-1, 1)).flatten()
        y_te_inv = scaler.inverse_transform(y_te_ws.reshape(-1, 1)).flatten()
        
        rmse = np.sqrt(mean_squared_error(y_te_inv, pred_inv))
        window_results[name].append(rmse)
        print(f"  {name:>4s}: RMSE = {rmse:.2f}")
        
        del model

# Plot results
fig, ax = plt.subplots(figsize=(12, 6))

for name, color, marker in [('RNN', '#2196F3', 'o'), ('LSTM', '#e53935', 's'), ('GRU', '#4CAF50', 'D')]:
    rmse_vals = window_results[name]
    ax.plot(window_sizes, rmse_vals, color=color, linewidth=2.5, marker=marker,
            markersize=10, label=name, zorder=3)
    
    # Annotate optimal window
    best_idx = np.argmin(rmse_vals)
    best_ws = window_sizes[best_idx]
    best_rmse = rmse_vals[best_idx]
    ax.annotate(f'Best: w={best_ws}\nRMSE={best_rmse:.1f}',
               xy=(best_ws, best_rmse),
               xytext=(15, -25), textcoords='offset points',
               fontsize=9, fontweight='bold', color=color,
               arrowprops=dict(arrowstyle='->', color=color, lw=1.5),
               bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor=color))

ax.set_xlabel('Window Size (months)', fontsize=13)
ax.set_ylabel('Test RMSE', fontsize=13)
ax.set_title('Window Size Sensitivity: RMSE vs Lookback Window', fontsize=16, fontweight='bold')
ax.set_xticks(window_sizes)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_facecolor('#FAFAFA')

# Highlight the seasonal period
ax.axvline(x=12, color='gray', linestyle=':', alpha=0.5)
ax.text(12.5, ax.get_ylim()[1]*0.95, 'Seasonal Period (12 mo)',
        fontsize=9, color='gray', style='italic')

plt.tight_layout()
plt.show()

## 8. Stacked Models (2-Layer) Comparison

Adding a second recurrent layer allows the model to learn **hierarchical temporal features**:
- **Layer 1** learns low-level patterns (local trends, short-term dependencies)
- **Layer 2** learns high-level abstractions (seasonal patterns, long-term trends)

The first recurrent layer must use `return_sequences=True` to pass a full sequence of hidden states to the second layer, rather than just the final hidden state.

In [ ]:
# ==== Stacked (2-Layer) Model Builders ====
# Each stacked model has 2 recurrent layers (first with return_sequences=True)
# followed by Dense(1) output. Same units=64, optimizer, loss as single-layer.
# ==================================================

def build_stacked_rnn(units=64, window=12):
    """Build a 2-layer stacked SimpleRNN model."""
    model = Sequential([
        SimpleRNN(units, return_sequences=True, input_shape=(window, 1)),
        SimpleRNN(units),
        Dense(1)
    ])
    model.compile(optimizer=Adam(0.001), loss='mse')
    return model

def build_stacked_lstm(units=64, window=12):
    """Build a 2-layer stacked LSTM model."""
    model = Sequential([
        LSTM(units, return_sequences=True, input_shape=(window, 1)),
        LSTM(units),
        Dense(1)
    ])
    model.compile(optimizer=Adam(0.001), loss='mse')
    return model

def build_stacked_gru(units=64, window=12):
    """Build a 2-layer stacked GRU model."""
    model = Sequential([
        GRU(units, return_sequences=True, input_shape=(window, 1)),
        GRU(units),
        Dense(1)
    ])
    model.compile(optimizer=Adam(0.001), loss='mse')
    return model

# Train stacked models
stacked_results = {}

print("Training Stacked (2-Layer) Models")
print("=" * 60)

for name, builder in [('RNN', build_stacked_rnn), ('LSTM', build_stacked_lstm), ('GRU', build_stacked_gru)]:
    print(f"\nTraining Stacked {name}...")
    
    np.random.seed(42)
    tf.random.set_seed(42)
    
    model = builder(units=64, window=WINDOW)
    
    start_time = time.time()
    history = model.fit(
        X_train, y_train,
        epochs=200, batch_size=16,
        validation_split=0.15,
        callbacks=[EarlyStopping(patience=20, restore_best_weights=True)],
        verbose=0
    )
    train_time = time.time() - start_time
    
    pred_test = model.predict(X_test, verbose=0).flatten()
    pred_test_inv = scaler.inverse_transform(pred_test.reshape(-1, 1)).flatten()
    y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
    
    rmse = np.sqrt(mean_squared_error(y_test_inv, pred_test_inv))
    r2 = r2_score(y_test_inv, pred_test_inv)
    
    stacked_results[name] = {
        'rmse': rmse, 'r2': r2,
        'time': train_time,
        'params': model.count_params(),
        'epochs': len(history.history['loss'])
    }
    
    print(f"  Params: {model.count_params():,}, Epochs: {stacked_results[name]['epochs']}")
    print(f"  RMSE: {rmse:.2f}, R²: {r2:.4f}, Time: {train_time:.1f}s")
    del model

# Comparison bar chart: Single vs Stacked
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
bar_colors = ['#2196F3', '#e53935', '#4CAF50']
model_labels = ['RNN', 'LSTM', 'GRU']
x = np.arange(len(model_labels))
width = 0.3

# Panel 1: RMSE comparison
single_rmse = [results[m]['rmse'] for m in model_labels]
stacked_rmse = [stacked_results[m]['rmse'] for m in model_labels]

for i, (sr, st, c) in enumerate(zip(single_rmse, stacked_rmse, bar_colors)):
    ax1.bar(x[i] - width/2, sr, width, color=c, alpha=0.6, edgecolor=c, linewidth=2, label='Single' if i==0 else '')
    ax1.bar(x[i] + width/2, st, width, color=c, alpha=1.0, edgecolor='black', linewidth=1, label='Stacked' if i==0 else '')

# Value annotations
for i in range(3):
    ax1.text(x[i] - width/2, single_rmse[i] + 0.5, f'{single_rmse[i]:.1f}',
             ha='center', fontsize=9, fontweight='bold')
    ax1.text(x[i] + width/2, stacked_rmse[i] + 0.5, f'{stacked_rmse[i]:.1f}',
             ha='center', fontsize=9, fontweight='bold')

ax1.set_xticks(x)
ax1.set_xticklabels(model_labels, fontsize=12)
ax1.set_ylabel('Test RMSE', fontsize=12)
ax1.set_title('RMSE: Single vs Stacked (2-Layer)', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
ax1.set_facecolor('#FAFAFA')

# Custom legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='gray', alpha=0.5, edgecolor='gray', label='Single Layer'),
                   Patch(facecolor='gray', alpha=1.0, edgecolor='black', label='Stacked (2-Layer)')]
ax1.legend(handles=legend_elements, fontsize=10)

# Panel 2: Parameter count comparison
single_params = [results[m]['params'] for m in model_labels]
stacked_params = [stacked_results[m]['params'] for m in model_labels]

for i, (sp, stp, c) in enumerate(zip(single_params, stacked_params, bar_colors)):
    ax2.bar(x[i] - width/2, sp, width, color=c, alpha=0.6, edgecolor=c, linewidth=2)
    ax2.bar(x[i] + width/2, stp, width, color=c, alpha=1.0, edgecolor='black', linewidth=1)

for i in range(3):
    ax2.text(x[i] - width/2, single_params[i] + 300, f'{single_params[i]:,}',
             ha='center', fontsize=8, fontweight='bold')
    ax2.text(x[i] + width/2, stacked_params[i] + 300, f'{stacked_params[i]:,}',
             ha='center', fontsize=8, fontweight='bold')

ax2.set_xticks(x)
ax2.set_xticklabels(model_labels, fontsize=12)
ax2.set_ylabel('Total Parameters', fontsize=12)
ax2.set_title('Parameters: Single vs Stacked', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
ax2.set_facecolor('#FAFAFA')
ax2.legend(handles=legend_elements, fontsize=10)

plt.suptitle('Single Layer vs Stacked (2-Layer) Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ==== Final Comparison Table ====
# Comprehensive DataFrame comparing all single-layer models across
# every metric: parameters, time, epochs, RMSE, MAE, R², MAPE.
# ==================================================

comparison = pd.DataFrame({
    'Model': ['RNN', 'LSTM', 'GRU'],
    'Parameters': [results[m]['params'] for m in ['RNN', 'LSTM', 'GRU']],
    'Training Time (s)': [round(results[m]['time'], 1) for m in ['RNN', 'LSTM', 'GRU']],
    'Epochs': [results[m]['epochs'] for m in ['RNN', 'LSTM', 'GRU']],
    'Test RMSE': [round(results[m]['rmse'], 2) for m in ['RNN', 'LSTM', 'GRU']],
    'Test MAE': [round(results[m]['mae'], 2) for m in ['RNN', 'LSTM', 'GRU']],
    'R\u00b2': [round(results[m]['r2'], 4) for m in ['RNN', 'LSTM', 'GRU']],
    'MAPE (%)': [round(results[m]['mape'], 1) for m in ['RNN', 'LSTM', 'GRU']],
})

print("=" * 80)
print("FINAL COMPARISON: RNN vs LSTM vs GRU (Single Layer, Window=12)")
print("=" * 80)
print(comparison.to_string(index=False))
print()

# Identify winners
best_rmse = comparison.loc[comparison['Test RMSE'].idxmin(), 'Model']
best_r2 = comparison.loc[comparison['R\u00b2'].idxmax(), 'Model']
fastest = comparison.loc[comparison['Training Time (s)'].idxmin(), 'Model']
smallest = comparison.loc[comparison['Parameters'].idxmin(), 'Model']

print(f"Best RMSE:        {best_rmse}")
print(f"Best R\u00b2:          {best_r2}")
print(f"Fastest Training: {fastest}")
print(f"Smallest Model:   {smallest}")

## 9. Final Analysis & Recommendations

### For the Airline Passengers Dataset:
Based on our experiments:

| Criterion | Best Model | Reason |
|-----------|-----------|--------|
| **Accuracy** | LSTM / GRU | Better at capturing long seasonal patterns |
| **Speed** | RNN | Fewest parameters, fastest per epoch |
| **Efficiency** | GRU | Near-LSTM accuracy with 25% fewer params |

### General Recommendations:

| Scenario | Recommendation |
|----------|---------------|
| Short sequences (< 20 steps) | **SimpleRNN** — simple, fast, sufficient |
| Medium sequences (20-100 steps) | **GRU** — good balance of accuracy and speed |
| Long sequences (> 100 steps) | **LSTM** — cell state highway handles long dependencies |
| Limited compute budget | **GRU** — 75% of LSTM's params |
| Maximum accuracy needed | **LSTM** — slight edge on complex patterns |
| Quick prototyping | **GRU** — fast to train, competitive results |

## Key Takeaways

1. **All three architectures** can learn the airline passenger trend and seasonality
2. **LSTM and GRU** typically outperform SimpleRNN on this dataset because 12-month seasonality requires remembering patterns from a year ago
3. **GRU** offers the best **accuracy-to-parameter ratio** — it's the "sweet spot" for many practical applications
4. **Window size matters**: All models benefit from a window of ~12 months (matching the seasonal period)
5. **Stacking layers** helps but with diminishing returns — 2 layers is usually the sweet spot

### Next Steps
- Try **Bidirectional** versions (see Session 4.2 notebook)
- Add **Dropout** for regularization (see S4_1, S5_1, S5_2 hyperparameter guides)
- Explore **Attention mechanisms** for even better performance

---

**Author:** Dr. Milan Joshi | MathCanvasMJ  
**Related Notebooks:**
- S4_1: RNN Hyperparameters Comprehensive
- S4_2: Bidirectional RNNs
- S5_1: LSTM Networks — From Plain to Fully Tuned
- S5_2: GRU Networks — From Plain to Fully Tuned

**References:**
- Hochreiter & Schmidhuber (1997): Long Short-Term Memory
- Cho et al. (2014): Learning Phrase Representations using RNN Encoder-Decoder
- Box & Jenkins (1970): Time Series Analysis, Forecasting and Control